# Day 1 · 01 · SQL Warehouse & Analyst Tooling

This notebook starts the RetailHub analytics story: work confidently in
Databricks SQL, know when to use SQL Editor versus a notebook, understand
what drives SQL Warehouse cost and latency, and discover the source data
before any Gold modelling begins.

## Business Scenario

RetailHub wants an executive KPI dashboard in Power BI, but today different
reports calculate revenue differently and nobody owns one approved source of
truth. Before building anything, the team first needs to work confidently in
Databricks SQL: know where to run queries, how warehouse choices affect cost
and speed, and what the source data actually looks like.

## Learning Objectives

By the end of this notebook you can:

- explain the analyst workflow in Databricks SQL,
- use SQL Editor, Catalog Explorer and Query History as analyst tools,
- decide when to use SQL Editor versus a notebook for a given task,
- identify what drives SQL Warehouse cost and latency,
- compare Serverless, Pro and Classic SQL Warehouses at decision level,
- use CTEs, temporary views and window functions to write readable BI SQL,
- recognize where SQL Scripting helps repeatable checks and automation,
- discover source grain and common data-quality risks before modelling Gold.

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this notebook.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before the demo starts.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream setup or demo notebook has not created the required tables.

In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: data/generate_training_dataset.ipynb")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"data/generate_training_dataset.ipynb\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## 1. SQL Warehouse orientation

![SQL Warehouse cost decision](../../assets/images/sql_warehouse_cost_decision.png)

Analysts usually experience Databricks through SQL Editor, notebooks, query
history and dashboards. For this course, the important operational idea is:
the warehouse is the compute layer behind SQL interactions.

| Area | What to show | Why it matters |
|---|---|---|
| SQL Editor | where ad-hoc SQL is written | fast exploration |
| Warehouse settings | size, type, auto-stop | direct cost impact |
| Query History | duration, user, warehouse | performance and cost diagnosis |
| Catalog Explorer | tables, comments, lineage | BI trust and discoverability |

`[UI DEMO]` Open SQL Warehouse settings and Query History before running the
next profiling queries.

### SQL Warehouse decision points for BI

A SQL Warehouse is not the data layer. It is the SQL/BI compute endpoint that
Power BI, Databricks SQL Editor, dashboards, JDBC/ODBC clients and the
Statement Execution API use to query governed Unity Catalog objects.

| Decision | What to inspect | BI impact |
|---|---|---|
| Warehouse type | Serverless, Pro, Classic | startup time, networking options, operational model |
| Size | S/M/L/XL and concurrency | query latency and user queueing |
| Auto-stop | idle timeout | idle DBU cost between report interactions |
| Query history | duration, rows, warehouse, user | report page cost and bottleneck diagnosis |
| Photon / Predictive IO | available warehouse capabilities | scan and join performance for SQL workloads |
| Access model | UC grants on catalog/schema/table | same governance for notebooks, SQL and Power BI |

Rule of thumb for this course: use the smallest warehouse that meets the BI
latency target, keep auto-stop tight, and make Power BI hit narrow Gold
objects instead of repeatedly scanning wide detail tables.

## 2. SQL Editor Productivity Lab

![SQL Editor](../../assets/images/source_sql_editor.png)

This is a short live demo, not a separate SQL Editor course. The goal is to
show how a Power BI developer or analyst works day to day before the modelling
starts.

For a Power BI developer, SQL Editor is the fastest way to answer practical
questions before a report is built:

| Question | SQL Editor action | Why it matters for Power BI |
|---|---|---|
| Which table should I connect to? | inspect catalog, schema and table comments | avoids connecting to a staging or Silver table by accident |
| What is the grain? | run count and distinct-count checks | prevents wrong relationships and inflated measures |
| Is a filter selective? | test date/status/channel predicates | predicts Import refresh size and DirectQuery latency |
| Is the query expensive? | open Query History after execution | links report design to warehouse cost |
| Can another developer reuse this logic? | save/query/comment the SQL | keeps BI logic outside individual PBIX files |

`[UI DEMO]` In SQL Editor, show:

1. **Catalog / schema picker** - select the training catalog and `silver` /
   `gold` schemas.
2. **Multi-statement results** - run two `SELECT` statements and compare their
   result tabs.
3. **Inline execution history** - find the last query and open its details.
4. **Databricks Assistant** - ask for a short explanation of a query, but do
   not let it replace validation.
5. **Default warehouse** - confirm which SQL Warehouse this analyst will use.

Trainer note: this 10-15 minute demo is what makes the course feel like a
real Databricks analyst workflow instead of only a notebook exercise.

### Hands-on: parameterized exploration query

The SQL Editor supports parameters. In a notebook we simulate the same idea
with a widget-backed value. Use this to show how analysts narrow a query before
they hand it to a dashboard.

In [ ]:
available_months = [
    r["year_month"]
    for r in spark.sql(f"""
        SELECT DISTINCT date_format(order_date, 'yyyy-MM') AS year_month
        FROM {SILVER}.sales_orders
        WHERE order_date IS NOT NULL
        ORDER BY year_month
        LIMIT 24
    """).collect()
]

default_month = available_months[-1] if available_months else "2025-01"

try:
    dbutils.widgets.dropdown("p_year_month", default_month, available_months or [default_month])
    p_year_month = dbutils.widgets.get("p_year_month")
except Exception:
    p_year_month = default_month

print("Selected month:", p_year_month)

spark.sql(f"""
SELECT
  date_format(o.order_date, 'yyyy-MM') AS year_month,
  o.channel,
  o.status,
  COUNT(DISTINCT o.order_id) AS orders
FROM {SILVER}.sales_orders o
WHERE date_format(o.order_date, 'yyyy-MM') = '{p_year_month}'
GROUP BY date_format(o.order_date, 'yyyy-MM'), o.channel, o.status
ORDER BY channel, status
""").show()

### SQL Editor vs notebook: which one for this task?

Both run the same Databricks SQL against the same SQL Warehouse. The choice
is about workflow, not capability.

| Task | Better fit | Why |
|---|---|---|
| Quick ad-hoc exploration, one or two statements | SQL Editor | fastest path, no cell/kernel overhead |
| Several transformation steps with narration for a workshop or handoff | Notebook | markdown + code together tells the story |
| Parameterized query you will re-run with different values | Either | SQL Editor parameters for ad-hoc, notebook widgets when the steps around it also change |
| Logic that should go through code review or live in a repo | Notebook | versionable, diff-able, reviewable |
| Repeated validation before a refresh or publish | Notebook or SQL Scripting | repeatable, can fail fast with a clear message |
| Sharing a single query result with a teammate right now | SQL Editor | saved query, shareable link, no notebook setup |

Rule of thumb for this course: SQL Editor for fast exploration and one-off
checks, notebook when the work has multiple steps, needs narration, or should
be reviewable as code.

## 3. SQL Programming Fundamentals for BI-ready Gold

SQL programming is the analyst layer between ad-hoc exploration and a stable
Gold model. The goal is not clever SQL. The goal is readable SQL that exposes
grain, filters and business rules before those rules become hidden in a PBIX.

| Building block | Use it for | BI relevance |
|---|---|---|
| CTE | name one query step | readable transformations before materializing Gold |
| Multi-CTE | filter -> aggregate -> rank pipeline | easier review of KPI logic |
| Window function | ranking, running totals, deduplication | top-N visuals, latest-record logic, SCD reasoning |
| Temp view | reuse a session-scoped shape | quick iterative analysis without creating UC objects |
| Session variable / dynamic SQL | parameterized exploration | safer demos and repeatable SQL Editor workflows |
| SQL UDF | shared expression-level rule | avoid copying business logic across reports |

In [ ]:
spark.sql(f"""
WITH completed_lines AS (
  SELECT
    customer_id,
    order_id,
    category,
    channel,
    line_revenue,
    line_margin
  FROM {SILVER}.order_lines
  WHERE status = 'Completed'
),
customer_revenue AS (
  SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS completed_orders,
    ROUND(SUM(line_revenue), 2) AS revenue,
    ROUND(SUM(line_margin), 2) AS gross_margin
  FROM completed_lines
  GROUP BY customer_id
),
ranked_customers AS (
  SELECT
    customer_id,
    completed_orders,
    revenue,
    gross_margin,
    DENSE_RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
  FROM customer_revenue
)
SELECT
  r.revenue_rank,
  r.customer_id,
  c.customer_name,
  c.segment,
  r.completed_orders,
  r.revenue,
  r.gross_margin
FROM ranked_customers r
JOIN {SILVER}.customers c ON c.customer_id = r.customer_id
WHERE r.revenue_rank <= 10
ORDER BY r.revenue_rank, r.revenue DESC
""").show(truncate=False)

### Reusable analysis shape: temporary view

A temporary view is useful when the same analysis shape is needed in several
cells during a live demo. It is not a Gold contract and it is not persisted in
Unity Catalog. If Power BI needs it repeatedly, convert it into a governed
Gold table or view.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW completed_sales_profile AS
SELECT
  category,
  channel,
  COUNT(DISTINCT order_id) AS completed_orders,
  ROUND(SUM(line_revenue), 2) AS revenue,
  ROUND(SUM(line_margin), 2) AS gross_margin,
  ROUND(100.0 * SUM(line_margin) / NULLIF(SUM(line_revenue), 0), 2) AS margin_rate_pct
FROM {SILVER}.order_lines
WHERE status = 'Completed'
GROUP BY category, channel
""")

spark.sql("""
SELECT *
FROM completed_sales_profile
ORDER BY revenue DESC
LIMIT 10
""").show(truncate=False)

## 4. SQL Scripting Fundamentals for repeatable analyst checks

SQL Scripting adds procedural control flow to SQL: variables, `IF`, loops and
compound `BEGIN ... END` blocks. In this training it is an orientation topic,
not the main implementation path. The practical value for analysts is knowing
when a repeated validation or parameterized SQL workflow should become a script
or job task instead of manual copy/paste.

Use cases that matter for BI teams:

- parameterized validation before a dashboard refresh,
- threshold checks before publishing a KPI,
- dynamic SQL for repeated schema/table profiling,
- job tasks that fail fast with readable messages,
- future stored procedures for governed operational SQL.

In [ ]:
try:
    spark.sql(f"""
BEGIN ATOMIC
  DECLARE completed_orders BIGINT DEFAULT 0;
  DECLARE completed_revenue DECIMAL(18,2) DEFAULT 0;

  SET completed_orders = (
    SELECT COUNT(DISTINCT order_id)
    FROM {SILVER}.sales_orders
    WHERE status = 'Completed'
  );

  SET completed_revenue = (
    SELECT ROUND(SUM(line_revenue), 2)
    FROM {SILVER}.order_lines
    WHERE status = 'Completed'
  );

  IF completed_orders > 0 THEN
    SELECT
      completed_orders,
      completed_revenue,
      ROUND(completed_revenue / completed_orders, 2) AS avg_order_value;
  ELSE
    SELECT
      completed_orders,
      completed_revenue,
      CAST(NULL AS DECIMAL(18,2)) AS avg_order_value;
  END IF;
END
""").show(truncate=False)
except Exception as e:
    print("[INFO] SQL Scripting compound statements are not available on this runtime/warehouse.")
    print("Use this cell as syntax orientation and run it live on a compatible Databricks SQL Warehouse.")
    print(type(e).__name__, str(e)[:300])

## 5. Pricing and warehouse decision

> Trainer TODO before delivery: capture a current official Databricks Pricing
> or Pricing Calculator screenshot and save it as
> `assets/images/databricks_pricing_YYYY-MM-DD.png`.

For Medi, keep this decision-level. Participants do not need to memorize every
price; they need to understand which choices change cost.

The practical pricing conversation has three parts:

1. **Unit price evidence** - official Databricks pricing or pricing calculator.
2. **Workload behavior** - how long the warehouse runs, how many users query it,
   and whether Power BI uses Import or DirectQuery.
3. **Model shape** - whether Power BI hits a narrow Gold aggregate or scans a
   wide detail table repeatedly.

| Warehouse type | Best use in this course | Watch out |
|---|---|---|
| Serverless | default for interactive SQL and BI demos | fastest start, but query volume still matters |
| Pro | controlled environments that need classic warehouse behavior | more operational choices |
| Classic | legacy or constrained setups | slower operational experience |

Core cost drivers:

- **DBU rate x runtime** - how long the warehouse is active.
- **Warehouse size** - larger warehouse costs more per active hour.
- **Auto-stop** - controls idle cost.
- **Concurrency** - many users and many Power BI visuals can create bursts.
- **Query shape** - bad SQL can cost more than the warehouse choice.

| Decision | Import mode tendency | DirectQuery tendency |
|---|---|---|
| Warehouse size | smaller is often enough | needs low latency |
| Auto-stop | aggressive auto-stop is fine | too aggressive can hurt user clicks |
| Query volume | scheduled refreshes | every visual and filter interaction |
| Best Gold source | aggregate table | narrow, selective Gold table/view |
| Cost risk | refresh window | user interaction fan-out |

### SQL Warehouse type decision matrix

This is the most important cost decision your team makes before the first dashboard goes live. The wrong type does not break analytics — it just costs more or starts slower than it should.

| Scenario | Recommended type | Key reason |
|---|---|---|
| 20–100 analysts writing ad-hoc queries, sporadic usage | **Serverless** | per-second billing → no idle cost between queries |
| Scheduled Gold refresh job running daily at 05:00 | **Pro** | per-minute billing → better for predictable long runs |
| Executive dashboard with DirectQuery and many concurrent users | **Serverless** | lowest cold-start (~2 s), handles concurrency automatically |
| BI app that needs a private endpoint or VNet injection | **Pro or Classic** | Serverless has limited network isolation options |
| Legacy environment or specific feature not yet on Serverless | **Classic** | only when Serverless/Pro are ruled out |

**Billing model — the critical difference:**

| Type | Billing unit | Cold start | Who manages cluster |
|---|---|---|---|
| Serverless | per **second** of actual query time | ~2 s | Databricks (you never see the cluster) |
| Pro | per **minute** the warehouse is running (including idle within auto-stop window) | ~20–40 s | Databricks (but you pick size and config) |
| Classic | per **minute** | ~60–120 s | you (cluster config, updates, networking) |

**Decision rule for this course:** Serverless is the right default for the analyst and BI workloads we build here. Choose Pro only when you have a long-running scheduled job or a specific networking requirement that Serverless cannot satisfy yet.

### Optional: cost monitoring orientation

If the workspace has access to system tables, show the idea of cost
monitoring. Do not make this a required lab, because permissions vary by
workspace.

In [ ]:
# Optional orientation only. This may require account-level/system table access.
try:
    spark.sql("""
    SELECT
      cloud,
      sku_name,
      usage_unit,
      ROUND(SUM(usage_quantity), 2) AS usage_quantity
    FROM system.billing.usage
    WHERE usage_start_time >= current_date() - INTERVAL 30 DAYS
    GROUP BY cloud, sku_name, usage_unit
    ORDER BY usage_quantity DESC
    LIMIT 10
    """).show(truncate=False)
except Exception as e:
    print("[INFO] system.billing.usage is not available in this workspace or for this user.")
    print("Use this as a UI discussion point instead of a required lab.")
    print(type(e).__name__, str(e)[:240])

## 6. Unity Catalog and Catalog Explorer orientation

![Catalog Explorer lineage](../../assets/images/source_catalog_explorer_lineage.png)

For this course, Unity Catalog is not an admin topic. It is the way analysts
discover trusted objects and understand ownership.

For Power BI developers, Catalog Explorer answers the questions that usually
get lost in report handoff:

| Catalog Explorer signal | What it tells the BI developer |
|---|---|
| table comment | whether this object is meant for BI consumption |
| column comments | which fields are safe for slicers, labels and measures |
| owner | who approves metric definitions or investigates data issues |
| lineage | which upstream tables feed a dashboard number |
| permissions | whether the report can be refreshed by the intended identity |

`[UI DEMO]` Show in Catalog Explorer:

1. catalog -> schema -> table hierarchy,
2. comments and table metadata,
3. lineage for a Gold object,
4. where a Power BI developer would find the object name to connect to.

In [ ]:
print("Active catalog and schema")
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)

print("Gold objects visible to this notebook")
spark.sql(f"SHOW TABLES IN {GOLD}").show(truncate=False)

## 7. Source data discovery

The first technical task is not to build Gold. It is to understand source
objects, grain and risk.

Discovery questions:

- What is the business entity?
- What is the grain?
- Which column is the stable key?
- Which columns are safe for filtering?
- Which values need business clarification?

In [ ]:
for table_name in ["customers", "products", "sales_orders", "order_lines"]:
    full_name = f"{SILVER}.{table_name}"
    rows = spark.sql(f"SELECT COUNT(*) AS n FROM {full_name}").first()["n"]
    print(f"{full_name}: {rows:,} rows")

In [ ]:
spark.sql(f"""
SELECT
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  COUNT(*) AS orders,
  COUNT(DISTINCT order_id) AS distinct_orders,
  COUNT(DISTINCT customer_id) AS distinct_customers
FROM {SILVER}.sales_orders
""").show(truncate=False)

spark.sql(f"""
SELECT status, COUNT(*) AS rows
FROM {SILVER}.sales_orders
GROUP BY status
ORDER BY rows DESC
""").show()

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS line_rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  ROUND(COUNT(*) / COUNT(DISTINCT order_id), 2) AS avg_lines_per_order
FROM {SILVER}.order_lines
""").show()

## 8. Data map and grain discussion

![RetailHub source map](../../assets/images/retailhub_source_map.png)

Fill this table as a group:

| Source object | Business entity | Grain | Key columns | Main risks |
|---|---|---|---|---|
| `silver.customers` | customer | one row per customer | `customer_id` | missing segment/region |
| `silver.products` | product | one row per product | `product_id` | missing category/cost |
| `silver.sales_orders` | order header | one row per order | `order_id` | status semantics |
| `silver.order_lines` | order line | one row per order-product line | `line_id` | joins, quantity, price |

### Common mistakes to catch early

| Mistake | Symptom later in BI | Prevention |
|---|---|---|
| Treating order lines as orders | inflated order count | use `COUNT(DISTINCT order_id)` |
| Joining dimensions before checking grain | duplicated revenue | validate keys and row counts before/after join |
| Building dashboard on Silver directly | duplicated KPI logic in reports | build Gold contract first |
| Ignoring status semantics | revenue includes cancelled or returned orders | agree KPI filters in Gold |
| Querying every column for BI | slow visuals and larger scans | select only columns required by report |

## Summary and next steps

Covered in this notebook:

- SQL Warehouse workflow and cost decision,
- SQL Editor productivity workflow, including when to use SQL Editor vs a notebook,
- SQL programming patterns: CTE, temp view, window ranking,
- SQL Scripting orientation for repeatable checks,
- Catalog Explorer / Unity Catalog orientation,
- source grain understanding and common mistakes to avoid.

Next: `day1_02_lakehouse_delta_gold.ipynb` builds on this discovery work to
explain the Lakehouse, Delta Lake foundations, and build the first trusted
Gold layer.